In [ ]:
import sys
import re
from pathlib import Path
import numpy as np
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore, QtWidgets
from scipy.signal import butter, sosfiltfilt, iirnotch, medfilt


HEADER_KV_RE = re.compile(r"^\s*([^:]+?)\s*:\s*(.*?)\s*$")
DATA_RE = re.compile(r"^\s*(\d{2}:\d{2}:\d{2}),(\d{3})\s*;\s*([+-]?\d+)\s*$")


def load_ecg_txt(path: str):
    """
    Parses files of the form:

    Signal Type: ECG2_Type
    Start Time: 29-05-2019 12:30:00
    Sample Rate: 256
    Length: 19353344
    Unit: µV

    Data:
    12:30:00,000; -4913
    ...

    Returns:
        fs (int): sampling rate (Hz)
        start_time_str (str|None): raw header Start Time value
        values_uv (np.ndarray float32): ECG samples in microvolts
    """
    header = {}
    values = []

    in_data = False
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if not in_data:
                if line.lower().startswith("data"):
                    in_data = True
                    continue

                m = HEADER_KV_RE.match(line)
                if m:
                    key = m.group(1).strip().lower()
                    val = m.group(2).strip()
                    header[key] = val
                continue

            # data section
            m = DATA_RE.match(line)
            if not m:
                continue

            v = int(m.group(3))
            values.append(v)

    if not values:
        raise ValueError(
            "No data samples found. Check that the file contains a 'Data:' section and valid rows."
        )

    # sampling rate
    fs_raw = header.get("sample rate", header.get("samplerate", "256"))
    try:
        fs = int(re.findall(r"\d+", fs_raw)[0])
    except Exception:
        fs = 256

    start_time_str = header.get("start time")

    values_uv = np.asarray(values, dtype=np.float32)  # microvolts
    return fs, start_time_str, values_uv


def notch_filter_sos(x, fs, f0=50.0, q=30.0):
    # f0: 50 or 60; q: notch sharpness (higher = narrower)
    w0 = f0 / (fs / 2.0)
    b, a = iirnotch(w0, q)
    # convert to SOS for stable filtfilt-like behavior
    # iirnotch returns (b, a); wrap as SOS via scipy's tf2sos if available
    from scipy.signal import tf2sos
    sos = tf2sos(b, a)
    return sosfiltfilt(sos, x.astype(np.float64)).astype(np.float32)


def ecg_bandpass_sos(x, fs, low=0.5, high=40.0, order=4):
    nyq = 0.5 * fs
    sos = butter(order, [low/nyq, high/nyq], btype="band", output="sos")
    y = sosfiltfilt(sos, x.astype(np.float64))
    return y.astype(np.float32)


x = self.values_uv_raw - float(self.values_uv_raw.mean())
# 1) Notch (choose 50 or 60)
x = notch_filter_sos(x, fs, f0=50.0, q=30.0)
# # 2) Bandpass
self.values_uv = ecg_bandpass_sos(x, fs, low=0.5, high=35.0, order=4)

window_uv = medfilt(window_uv, kernel_size=3)  # in update_plot, after slicing


class ECGViewer(QtWidgets.QMainWindow):
    def __init__(self, fs: int, values_uv: np.ndarray, title: str):
        super().__init__()

        self.fs = fs

        # Window size
        self.window_seconds = 10.0
        self.window_samples = int(self.window_seconds * self.fs)

        # Store RAW
        self.values_uv_raw = values_uv.astype(np.float32)

        # Optional: filter the entire signal once (recommended)
        #x = self.values_uv_raw - float(self.values_uv_raw.mean())
        #self.values_uv = ecg_bandpass_sos(x, fs)  # displayed signal

        # ---- UI ----
        central = QtWidgets.QWidget()
        self.setCentralWidget(central)
        layout = QtWidgets.QVBoxLayout(central)

        self.plot = pg.PlotWidget()
        self.plot.setTitle(title)
        self.plot.setLabel("bottom", "Time", units="s")
        self.plot.setLabel("left", "Amplitude", units="mV")
        self.plot.showGrid(x=True, y=True, alpha=0.3)
        self.curve = self.plot.plot(pen=pg.mkPen(width=1))
        layout.addWidget(self.plot)

        # Scrollbar
        self.scroll = QtWidgets.QScrollBar(QtCore.Qt.Orientation.Horizontal)
        self.scroll.setMinimum(0)
        self.scroll.setMaximum(max(0, len(self.values_uv) - self.window_samples))
        self.scroll.setSingleStep(self.fs)            # 1 second
        self.scroll.setPageStep(self.window_samples)  # 10 seconds
        self.scroll.valueChanged.connect(self.update_plot)
        layout.addWidget(self.scroll)

        # Info label (THIS fixes your error)
        self.info = QtWidgets.QLabel()
        layout.addWidget(self.info)

        self.resize(1200, 600)

        # Start at 5 minutes by setting the scrollbar (instead of calling update_plot twice)
        start_idx = min(5 * 60 * self.fs, self.scroll.maximum())
        self.scroll.setValue(start_idx)  # triggers update_plot automatically

    def update_plot(self, start_idx: int):
        end_idx = min(start_idx + self.window_samples, len(self.values_uv))
        window_uv = self.values_uv[start_idx:end_idx]

        # µV -> mV
        window_mv = window_uv / 1000.0
        t = np.arange(len(window_mv), dtype=np.float32) / float(self.fs)

        self.curve.setData(t, window_mv)
        self.plot.setXRange(0, self.window_seconds, padding=0)

        start_sec = start_idx / float(self.fs)
        end_sec = (end_idx - 1) / float(self.fs)
        self.info.setText(
            f"fs={self.fs} Hz | samples={len(self.values_uv):,} | showing {start_sec:.2f}–{end_sec:.2f} s"
        )


def find_ecg_files(root_folder: str):
    """
    Recursively finds files named exactly 'ecg.txt' under root_folder.
    """
    root = Path(root_folder)
    return sorted(root.rglob("ecg.txt"))


def main():
    ROOT_FOLDER = r"C:\Users\JLOR0029\Desktop\Patients_ECG"

    app = QtWidgets.QApplication(sys.argv)

    ecg_files = find_ecg_files(ROOT_FOLDER)

    if not ecg_files:
        QtWidgets.QMessageBox.warning(
            None,
            "No Files Found",
            "No ecg.txt files found in the selected folder (search is recursive)."
        )
        sys.exit(0)

    # If multiple ecg.txt files are found, let the user choose one
    if len(ecg_files) > 1:
        items = [str(f) for f in ecg_files]
        chosen, ok = QtWidgets.QInputDialog.getItem(
            None,
            "Select ECG File",
            "Choose ecg.txt to view:",
            items,
            0,
            False
        )
        if not ok:
            sys.exit(0)
        selected_file = chosen
    else:
        selected_file = str(ecg_files[0])

    fs, start_time_str, values_uv = load_ecg_txt(selected_file)

    title = f"{Path(selected_file).parent.name} | {fs} Hz"
    if start_time_str:
        title += f" | Start: {start_time_str}"

    viewer = ECGViewer(fs=fs, values_uv=values_uv, title=title)
    viewer.show()
    
    app.exec()


if __name__ == "__main__":
    main()
